# Tested associations

A profile does not only state the associations it finds. Every pair
the association screens compute statistics for is kept, with the
test the screen ran on it, and `report.associations()` returns the
whole set as one table: the pair, its kind, its effect sizes, its
raw and adjusted p-values, and its tier. This page is about reading
that table — what each screen tests, how the correction is scoped,
what the two statement tiers mean, why some pairs carry an effect
size and no p-value, and what the Simpson reversal check adds to
the strong statements.

The worked example is the Adult census table from the benchmark
corpus (48,842 rows, 15 columns), profiled at the standard budget.
Two neighbouring pages cover adjacent ground: [Nominal
associations](associations.ipynb) describes the rule miner over
the same columns, and [Dependence measures](dependence_measures.ipynb)
defines the numeric coefficients the first screen uses.

In [1]:
import polars as pl

import onceover

df = pl.read_csv(
    "../../../benchmarks/corpus/data/business/adult-census-income/adult-census-income_uci-2.csv"
)
report = onceover.profile(df, budget="standard")

strong = [
    f for f in report.findings
    if f["category"] == "associations" and f["type"] == "statistical"
    and f["evidence"][0]["metrics"].get("tier") == "strong"
]
print(f"{df.height} rows, {df.width} columns; {len(strong)} strong association statements:")
for f in strong:
    print("-", f["claim"])

48842 rows, 15 columns; 4 strong association statements:
- education_num differs across the levels of education (correlation ratio 1.000)
- relationship and sex show nominal association (Cramer's V 0.650)
- age differs across the levels of marital_status (correlation ratio 0.575)
- education_num differs across the levels of occupation (correlation ratio 0.563)


The four statements are pooled claims about the evaluated
rows. The first restates a structure the report also records two
other ways — `education` and `education_num` are the same grouping
in two encodings, which the report states as an exact functional
dependency in both directions and as a duplicate encoding; the
pair screen's own reading of it is the correlation ratio of 1.000.
The second pairs two nominal columns. The third and fourth say a
numeric column's average differs across a nominal column's levels,
by enough that the between-level share of its variance clears one
half.

Each statement is unconditional: it describes all the evaluated
rows at once, with no subgroup named. That property is what the
Simpson check at the end of this page guards. First, the table
underneath the statements.

In [2]:
assoc = report.associations()
print(f"{assoc.height} pairs in the associations table")
outcome = (
    assoc.with_columns(
        pl.when(pl.col("p_value").is_null()).then(pl.lit("untested"))
        .otherwise(pl.col("tier")).alias("outcome")
    )
    .group_by(["kind", "outcome"]).len()
    .sort(["kind", "outcome"])
)
print(outcome)

91 pairs in the associations table
┌─────────┬──────────┬─────┐
│ kind    ┆ outcome  ┆ len │
╞═════════╪══════════╪═════╡
│ mixed   ┆ none     ┆ 2   │
│ mixed   ┆ strong   ┆ 3   │
│ mixed   ┆ weak     ┆ 43  │
│ nominal ┆ strong   ┆ 1   │
│ nominal ┆ untested ┆ 16  │
│ nominal ┆ weak     ┆ 11  │
│ numeric ┆ none     ┆ 2   │
│ numeric ┆ weak     ┆ 13  │
└─────────┴──────────┴─────┘


In [3]:
import altair as alt

chart_data = (
    assoc.with_columns(
        pl.when(pl.col("p_value").is_null()).then(pl.lit("untested"))
        .otherwise(pl.col("tier")).alias("outcome")
    )
    .group_by(["kind", "outcome"]).len()
)
(alt.Chart(chart_data).mark_bar()
 .encode(x=alt.X("kind:N", title="Pair kind"),
         y=alt.Y("len:Q", title="Pairs"),
         color=alt.Color("outcome:N", title="Outcome",
                         scale=alt.Scale(domain=["strong", "weak", "none", "untested"],
                                         range=["#1b7837", "#7fbc41", "#bdbdbd", "#d7301f"])))
 .properties(title="The screen's output, by pair kind and outcome", width=500, height=300))

alt.Chart(...)

What to look for: strong statements are four pairs out of
91. Most of the screen's output is weak or unstated, and the
nominal column carries a block the other two kinds do not have —
pairs holding an effect size and no test at all.

## What the table holds

One row per pair a screen computed statistics for, whether or not
the pair earned a statement. The columns are the pair (`a`, `b`),
its `kind` (`numeric`, `nominal`, or `mixed`), the pair-complete
row count `n` — rows where both sides are present, on the rows the
screen evaluated — the effect sizes that screen computes, the raw
`p_value`, the adjusted `q_value`, and the `tier`: `strong`,
`weak`, or `none`.

The row count needs one qualification. At the standard budget the
screens evaluate a sample of at most 20,000 rows, so on this table
`n` counts pair-complete rows within that sample: pairs involving
`occupation` rest on 18,837 rows because the column's missing
values remove the rest. Pairs involving a column the frame model
withholds as scaffolding are carried in the table like any other
pair; they earn no finding.

## What each screen tests

Three screens feed the table, and each pair kind has its own test
and its own effect size.

**Numeric pairs** are tested by the dependence screen. Each pair
gets a p-value for Pearson's r and for Spearman's ρ through the
standard t approximation, and one for Chatterjee's ξ in each
direction through the permutation distribution of its rank
statistic — the permutation mean and variance have closed forms,
so the ξ test costs no simulated permutations. The pair's p-value
is the smallest of the component p-values, multiplied by the
number of components computed: a Bonferroni combination, valid
whatever the dependence between components. A component that is
undefined — ξ on a constant side, or on fewer than 30 pairs — is
simply not among the components combined.

**Nominal pairs** are tested by the pair screen with a G-test on
the pair-complete contingency table; the effect size is Cramér's
V. Nominal columns enter with at most 16 observed levels.

**Mixed pairs** — a nominal column against a numeric one — are
tested by the same screen with a one-way F-test of the numeric
column across the nominal column's levels; the effect size is the
correlation ratio η, the share of the numeric column's variance
that sits between levels.

A pair is tested only when its inputs support a test: at least 30
pair-complete rows, no constant side, and, for the G-test, a
smallest expected cell count of at least 1. The last condition
decides several pairs on this table, in its own section below.

## The correction is per screen

Testing dozens of pairs at a 5% bar manufactures discoveries by
chance, so each screen adjusts its own family with
Benjamini–Hochberg: the numeric screen over the numeric pairs it
tested in the run, the pair screen over the nominal and mixed
pairs it tested together. The coverage entries state both family
sizes, and they are part of reading any q-value — a q is a
statement about a pair inside its family, not a property of the
pair alone.

In [4]:
for key in ("dependence_screen", "association_pairs"):
    print(key, "->", report.data["coverage"][key]["detail"])

dependence_screen -> 15 of 15 numeric pairs (full); all 15 pairs screened; detailed statistics over 20000 of 48842 rows; 0 strong pairwise, 13 weak pairwise (Benjamini-Hochberg q <= 0.05 over 15 tested pairs), 0 redundancy links
association_pairs -> 28 nominal pairs, 48 mixed pairs screened over 20000 encoded rows; 60 tested with p-values, Benjamini-Hochberg over the 60 tested pairs: 4 strong and 16 weak pair findings (q <= 0.05, effect bar 0.5); untested pairs are carried in the associations table


On this table the numeric family is all 15 pairs — every
pair received detailed statistics, so no first-stage cap cut the
family down. (On wider tables the dependence screen details only
the survivors of a binned mutual-information pre-screen, and the
coverage entry states the smaller family it corrected over.) The
pair screen's family is 60 of its 76 pairs; the remaining 16 were
computed but not tested, and untested pairs do not enter the
correction.

## The two tiers

A tested pair is stated under one rule, which the cell below
recomputes from the table itself: q ≤ 0.05 with the headline
effect at or above 0.5 is `strong`; q ≤ 0.05 below the bar is
`weak`; anything else is `none`. The headline effect is the
largest of |Pearson|, |Spearman| and ξ in both directions for a
numeric pair, and V or η for the pair screens.

In [5]:
headline = (
    pl.when(pl.col("kind") == "numeric")
      .then(pl.max_horizontal(pl.col("pearson").abs(), pl.col("spearman").abs(),
                              pl.col("xi_ab"), pl.col("xi_ba")))
      .when(pl.col("kind") == "nominal").then(pl.col("cramers_v"))
      .otherwise(pl.col("eta"))
)
rule = (
    pl.when((pl.col("q_value") <= 0.05) & (pl.col("effect") >= 0.5)).then(pl.lit("strong"))
      .when(pl.col("q_value") <= 0.05).then(pl.lit("weak"))
      .otherwise(pl.lit("none"))
)
tested = assoc.filter(pl.col("p_value").is_not_null()).with_columns(headline.alias("effect"))
agrees = tested.with_columns(rule.alias("expected")).filter(pl.col("tier") == pl.col("expected"))
print(f"the rule reproduces the recorded tier on {agrees.height} of {tested.height} tested pairs")

the rule reproduces the recorded tier on 75 of 75 tested pairs


The split the rule makes is between significance and size,
and at 20,000 rows the two come apart. `marital_status` × `sex`
has Cramér's V of 0.463 with p and q both recorded as 0.0 — the
G-test tail is smaller than a float can hold — and it is weak,
because 0.463 is below the bar. `final_weight` × `hours_per_week`
is weak with a largest measured effect of 0.032 and q ≈ 1.6×10⁻⁹:
the test detects a dependence far below any size worth acting
on, and states it anyway, with its evidence attached. The tier,
not the q-value, carries the size judgment.

Statements are capped per screen — at most 20 from each — while
the table is not capped. This report holds 33 association
findings: the 4 strong pairs, 16 of the pair screen's weak pairs,
and all 13 of the numeric screen's weak pairs. In the digest the
weak findings collapse into a single line:

In [6]:
weak_lines = [line for line in report.to_digest(20_000).splitlines() if "weak" in line]
for line in weak_lines:
    print(line[:280])

[associations] statistical/weak (x29): 29 tested association(s) below the strong effect bar — strongest: age differs weakly across the levels of relationship (correlation ratio 0.470; significant after Benjamini-Hochberg correction, q = <0.001); marital_status and sex show weak n


In [7]:
tested_pairs = tested.with_columns(
    pl.when(pl.col("q_value") > 0)
      .then(-pl.col("q_value").log10())
      .otherwise(320.0)
      .alias("neglog10_q")
)
points = alt.Chart(tested_pairs).mark_circle(size=42, opacity=0.65).encode(
    x=alt.X("effect:Q", title="Headline effect", scale=alt.Scale(domain=[0, 1.05])),
    y=alt.Y("neglog10_q:Q", title="-log10 q (q = 0 drawn at 320)"),
    color=alt.Color("tier:N", title="Tier",
                    scale=alt.Scale(domain=["strong", "weak", "none"],
                                    range=["#1b7837", "#7fbc41", "#bdbdbd"])),
    tooltip=["a", "b", "kind", "tier"],
)
bar = alt.Chart().mark_rule(color="#d7301f").encode(x=alt.datum(0.5))
(points + bar).properties(
    title="Tested pairs: effect size against adjusted significance", width=520, height=320)

alt.LayerChart(...)

What to look for: the four strong pairs sit at the right,
past the 0.5 line. A tall column of weak pairs reaches the top of
the chart at small effects — at this sample size significance is
cheap, and the red line is the whole difference between the
tiers. Pairs recorded with q = 0.0 are drawn at 320; their tails
underflow the float format rather than ending there.

One numeric pair shows why the combination rule matters.
`age` × `final_weight` has Pearson r of −0.070 and ξ(age →
final_weight) of 0.001, but ξ(final_weight → age) of 0.176 — the
largest component, and the one the claim names. ξ is directional:
knowing the census weight narrows a respondent's age
appreciably, while knowing the age barely narrows the weight.
Any one component can carry a pair's test; the Bonferroni
multiplier is the price of having four chances to do so.

## When a pair carries no p-value

Sixteen of the 28 nominal pairs hold a Cramér's V and a tier of
`none`, with `p_value` and `q_value` empty. They were not tested
and found clear; they were not tested. The G-test is withheld
when the smallest expected cell count of the pair's contingency
table falls below 1 on the evaluated rows — the chi-square tail
behind the test is not a reading of a table that sparse.

The largest of them is `marital_status` × `relationship`, V =
0.490, a hair under the strong bar. The two columns are closely
tied — `Husband` and `Wife` occur almost only inside
`Married-civ-spouse` — so the withheld test is not a verdict of
no association. What decides it is one rare level:
`Married-AF-spouse` holds 37 of the table's 48,842 rows.

In [8]:
pair_row = assoc.filter((pl.col("a") == "marital_status") & (pl.col("b") == "relationship"))
print(pair_row.select(["a", "b", "n", "cramers_v", "p_value", "q_value", "tier"]))

sub = df.select(["marital_status", "relationship"]).drop_nulls()
counts = (sub.group_by(["marital_status", "relationship"]).len()
          .pivot(on="relationship", index="marital_status", values="len").fill_null(0))
print(counts)
import numpy as np
cols = [c for c in counts.columns if c != "marital_status"]
M = counts.select(cols).to_numpy().astype(float)
E = M.sum(1, keepdims=True) @ M.sum(0, keepdims=True) / M.sum()
print(f"on all {int(M.sum())} rows the smallest expected cell count is {E.min():.2f}")

┌────────────────┬──────────────┬───────┬───────────┬─────────┬─────────┬──────┐
│ a              ┆ b            ┆ n     ┆ cramers_v ┆ p_value ┆ q_value ┆ tier │
╞════════════════╪══════════════╪═══════╪═══════════╪═════════╪═════════╪══════╡
│ marital_status ┆ relationship ┆ 20000 ┆ 0.489756  ┆ null    ┆ null    ┆ none │
└────────────────┴──────────────┴───────┴───────────┴─────────┴─────────┴──────┘
┌───────────────────────┬───────────────┬─────────┬──────┬───────────┬───────────┬────────────────┐
│ marital_status        ┆ Not-in-family ┆ Husband ┆ Wife ┆ Own-child ┆ Unmarried ┆ Other-relative │
╞═══════════════════════╪═══════════════╪═════════╪══════╪═══════════╪═══════════╪════════════════╡
│ Never-married         ┆ 7114          ┆ 0       ┆ 0    ┆ 6750      ┆ 1333      ┆ 920            │
│ Married-civ-spouse    ┆ 23            ┆ 19704   ┆ 2308 ┆ 143       ┆ 0         ┆ 201            │
│ Divorced              ┆ 3628          ┆ 0       ┆ 0    ┆ 455       ┆ 2369      ┆ 181         

on all 48842 rows the smallest expected cell count is 1.14


On the full table the smallest expected count is 1.14 — just
above the bar. The screen does not evaluate the full table: at
the standard budget it works on 20,000 sampled rows, where the
same margins give the rarest level about 15 rows and its
smallest expected counts fall below 1. Scaled to the evaluated
rows, the expected table is:

In [9]:
expected = pl.DataFrame(
    [{"marital_status": counts["marital_status"][i], "relationship": c,
      "expected": float(E[i, j] * 20_000 / M.sum())}
     for i in range(M.shape[0]) for j, c in enumerate(cols)]
)
(alt.Chart(expected).mark_rect()
 .encode(x=alt.X("relationship:N", title=None),
         y=alt.Y("marital_status:N", title=None, sort="-x"),
         color=alt.Color("expected:Q", title="Expected rows",
                         scale=alt.Scale(scheme="blues")))
 .properties(title="Expected cell counts at the screen's 20,000 evaluated rows",
             width=520, height=300))

alt.Chart(...)

What to look for: the `Married-AF-spouse` row is thin
throughout, and its crossings with `Wife` and `Other-relative`
fall below one expected row. One cell under the bar withholds
the pair's test.

The check is rerun below at the deep budget, restricted to the
pair screen. Deep evaluates all 48,842 rows, so the pair's
smallest expected count is the full table's 1.14, and the test
runs:

In [10]:
deep = onceover.profile(df, budget="deep", detectors=["association_rules"])
deep_assoc = deep.associations()
for pair in [("marital_status", "relationship"), ("work_class", "education")]:
    row = deep_assoc.filter((pl.col("a") == pair[0]) & (pl.col("b") == pair[1]))
    print(row.select(["a", "b", "n", "cramers_v", "p_value", "q_value", "tier"]))
print("work_class level counts, smallest three:")
print(df["work_class"].value_counts().sort("count").head(3))

┌────────────────┬──────────────┬───────┬───────────┬─────────┬─────────┬──────┐
│ a              ┆ b            ┆ n     ┆ cramers_v ┆ p_value ┆ q_value ┆ tier │
╞════════════════╪══════════════╪═══════╪═══════════╪═════════╪═════════╪══════╡
│ marital_status ┆ relationship ┆ 48842 ┆ 0.48816   ┆ 0.0     ┆ 0.0     ┆ weak │
└────────────────┴──────────────┴───────┴───────────┴─────────┴─────────┴──────┘
┌────────────┬───────────┬───────┬───────────┬─────────┬─────────┬──────┐
│ a          ┆ b         ┆ n     ┆ cramers_v ┆ p_value ┆ q_value ┆ tier │
╞════════════╪═══════════╪═══════╪═══════════╪═════════╪═════════╪══════╡
│ work_class ┆ education ┆ 46043 ┆ 0.102081  ┆ null    ┆ null    ┆ none │
└────────────┴───────────┴───────┴───────────┴─────────┴─────────┴──────┘
work_class level counts, smallest three:
┌──────────────┬───────┐
│ work_class   ┆ count │
╞══════════════╪═══════╡
│ Never-worked ┆ 10    │
│ Without-pay  ┆ 21    │
│ Federal-gov  ┆ 1432  │
└──────────────┴───────┘


Given the rows, the pair is tested and lands weak: V = 0.488
with q = 0.0. Its tier changed because the evidence behind it
changed, not because the association moved — the two V readings
differ by 0.002. `work_class` × `education` stays untested even
on every row: `work_class` has a level, `Never-worked`, with 10
rows in the whole census, and no budget repairs a margin that
thin. An untested row is a statement about the table's margins.
Reading it as "no association" is the one misreading this table
invites, and the empty p-value is what marks the difference.

## The Simpson reversal check

A strong statement is unconditional, so it can be wrong in a
specific way: the pooled direction can reverse inside every
subgroup of some other column. The classic shape is a treatment
that looks better overall and worse inside every severity band,
because the bands were assigned unevenly. Every
strong statement with a signed effect is checked for that shape
before the report is finished.

The check reads only signed effects. For a numeric pair, the
signed effect is the larger of Pearson's r and Spearman's ρ,
and the pair is checked only when that component reaches 0.3 in
absolute value — below it, the pair's strength is the unsigned
ξ and there is no material direction to reverse. For a mixed
pair whose nominal side has exactly two levels, it is the
point-biserial correlation. For a nominal pair that is 2×2 on
its observed levels, it is the log odds ratio. Cramér's V beyond
the 2×2 case, η over more than two levels, and ξ have no sign,
so pairs carried by them are not checked.

A stratifier is another categorical column with 2 to 8 observed
levels, not one of the pair's own columns and not withheld as
scaffolding. A stratum is usable with at least 30 pair-complete
rows and a defined effect, and the usable strata must together
cover at least half of the pair's rows. The stratifier
qualifies the statement when every usable stratum's effect has
the opposite sign to the pooled effect, and the combined
within-stratum effect — a Fisher-z average for correlations,
inverse-variance weighted for odds ratios — differs from zero
at |z| ≥ 3.84. The z bar is a Bonferroni bar at family α = 0.05
over the check's budget of 400 pair×stratifier evaluations per
screen; at most 24 stratifiers are considered per pair, and
when a bound stops a check the screen's coverage entry says so.
A qualification is appended to the claim and recorded in the
evidence as a `simpson` payload. The finding keeps its tier,
and the associations table is unchanged.

On the Adult table the check's scope is empty. All four strong
pairs are carried by unsigned effects:

In [11]:
levels = {c: df[c].drop_nulls().n_unique()
          for c in ["education", "marital_status", "occupation", "relationship"]}
accounting = pl.DataFrame([
    {"pair": "education x education_num", "kind": "mixed", "effect": "eta = 1.000",
     "signed_reading": "none: eta over 16 levels"},
    {"pair": "relationship x sex", "kind": "nominal", "effect": "V = 0.650",
     "signed_reading": "none: V on a 6x2 table"},
    {"pair": "marital_status x age", "kind": "mixed", "effect": "eta = 0.575",
     "signed_reading": "none: eta over 7 levels"},
    {"pair": "occupation x education_num", "kind": "mixed", "effect": "eta = 0.563",
     "signed_reading": "none: eta over 14 levels"},
])
print(accounting)
reversed_claims = [f for f in report.findings if "direction reverses" in f["claim"]]
print(f"claims carrying a reversal qualification: {len(reversed_claims)}")

┌────────────────────────────┬─────────┬─────────────┬──────────────────────────┐
│ pair                       ┆ kind    ┆ effect      ┆ signed_reading           │
╞════════════════════════════╪═════════╪═════════════╪══════════════════════════╡
│ education x education_num  ┆ mixed   ┆ eta = 1.000 ┆ none: eta over 16 levels │
│ relationship x sex         ┆ nominal ┆ V = 0.650   ┆ none: V on a 6x2 table   │
│ marital_status x age       ┆ mixed   ┆ eta = 0.575 ┆ none: eta over 7 levels  │
│ occupation x education_num ┆ mixed   ┆ eta = 0.563 ┆ none: eta over 14 levels │
└────────────────────────────┴─────────┴─────────────┴──────────────────────────┘
claims carrying a reversal qualification: 0


So the check examines none of the four, and no qualification
appears. That silence needs care: it means the strong statements
on this table are outside the check's scope, not that the check
examined them and cleared them. The report is equally silent
when eligible pairs are examined and cleared — coverage mentions
the check only when a qualification is attached or a bound stops
it. What the reader can rely on is the conditional: if a strong
statement on a signed pair reverses inside every usable stratum
of some recorded dimension, at a measured within effect, the
claim itself carries the qualification.

## What a qualification looks like

No bundled corpus table produces one — across the corpus the
criterion, reversal inside every usable stratum, is met by no
strong pair. The shape of a qualified statement is therefore
shown on a planted frame, the same design the test suite uses as
its regression fixture: two groups of 2,000 rows; inside each
group y falls as x rises; the second group sits higher on both
axes, so the pooled correlation comes out positive.

In [12]:
import random

rng = random.Random(7)
x, y, g = [], [], []
for grp, (lo, base) in enumerate([(0.0, 0.0), (3.0, 8.0)]):
    for _ in range(2000):
        xi = lo + 2.0 * rng.random()
        x.append(xi)
        y.append(base - 1.5 * xi + rng.gauss(0.0, 0.3))
        g.append("a" if grp == 0 else "b")
planted = pl.DataFrame({"x": x, "y": y, "g": g})

planted_report = onceover.profile(planted)
for f in planted_report.findings:
    if f["type"] == "statistical" and set(f["subject"]["columns"]) == {"x", "y"}:
        print(f["claim"])
        s = f["evidence"][0]["metrics"]["simpson"]
        print(pl.DataFrame(s["strata"]))
        print({k: s[k] for k in ("stratifier", "statistic", "pooled_effect",
                                 "within_effect", "within_z", "rows_covered_share")})

x and y show numeric association (maximum measured effect 0.846); the direction reverses within every level of g (pooled Pearson r +0.671; within-level Pearson r from -0.944 to -0.944, combined -0.944)
┌───────────┬───────┬──────┐
│ effect    ┆ level ┆ n    │
╞═══════════╪═══════╪══════╡
│ -0.944429 ┆ a     ┆ 2000 │
│ -0.944174 ┆ b     ┆ 2000 │
└───────────┴───────┴──────┘
{'stratifier': 'g', 'statistic': 'pearson', 'pooled_effect': 0.6712518456479284, 'within_effect': -0.9443015083765381, 'within_z': -112.26210524497714, 'rows_covered_share': 1.0}


In [13]:
pooled_line = (alt.Chart(planted).mark_line(color="black", strokeDash=[5, 4])
               .transform_regression("x", "y").encode(x="x:Q", y="y:Q"))
group_lines = (alt.Chart(planted).mark_line()
               .transform_regression("x", "y", groupby=["g"])
               .encode(x="x:Q", y="y:Q", color=alt.Color("g:N", title="Group")))
points = (alt.Chart(planted).mark_circle(size=14, opacity=0.25)
          .encode(x="x:Q", y="y:Q", color=alt.Color("g:N", title="Group")))
(points + group_lines + pooled_line).properties(
    title="Planted frame: pooled trend (dashed) against the two group trends",
    width=520, height=320)

alt.LayerChart(...)

What to look for: the dashed pooled line climbs, because
group b sits up and to the right of group a. Both group lines
fall. The pooled statement is true of the pooled rows and false
of every group in them.

The claim states the qualification in one clause — the
stratifier, the pooled effect, the range of within-stratum
effects, and the combined within effect, all with signs. The
payload behind it carries the same quantities per stratum, the
z of the combined effect (−112.3 here, far past the 3.84 bar),
and the share of the pair's rows the usable strata cover (1.0).
The finding is otherwise untouched: same tier, same evidence,
and its row in the associations table carries no trace of the
qualification — the table records the screen, the qualification
lives on the statement.

In [14]:
print(planted_report.associations().filter(
    (pl.col("a") == "x") & (pl.col("b") == "y")
).select(["a", "b", "kind", "pearson", "spearman", "p_value", "q_value", "tier"]))

┌───┬───┬─────────┬──────────┬──────────┬─────────┬─────────┬────────┐
│ a ┆ b ┆ kind    ┆ pearson  ┆ spearman ┆ p_value ┆ q_value ┆ tier   │
╞═══╪═══╪═════════╪══════════╪══════════╪═════════╪═════════╪════════╡
│ x ┆ y ┆ numeric ┆ 0.671252 ┆ 0.51136  ┆ 0.0     ┆ 0.0     ┆ strong │
└───┴───┴─────────┴──────────┴──────────┴─────────┴─────────┴────────┘


## The case the z bar exists for

Opposite signs inside the strata are not by themselves a
reversal. A pooled correlation can also be manufactured by
group means alone, with x and y independent inside each group:
then the within-stratum effects are noise around zero, and with
two strata both land on the opposite side of the pooled sign
about a quarter of the time. The frame below is built that way —
the same design the test suite uses — with a shift of 2 on both
columns between the two levels of `g` and no relationship inside
either level.

In [15]:
rng = random.Random(0)
x, y, g = [], [], []
for _ in range(2000):
    grp = rng.random() < 0.5
    x.append(2.0 * grp + rng.gauss(0.0, 1.0))
    y.append(2.0 * grp + rng.gauss(0.0, 1.0))
    g.append("B" if grp else "A")
guard = pl.DataFrame({"x": x, "y": y, "g": g})

guard_report = onceover.profile(guard)
for f in guard_report.findings:
    if f["type"] == "statistical" and set(f["subject"]["columns"]) == {"x", "y"}:
        print(f["claim"])
print("pooled Pearson r:", guard.select(pl.corr("x", "y")).item())
for lv in ("A", "B"):
    sub = guard.filter(pl.col("g") == lv)
    print(f"within level {lv}: r = {sub.select(pl.corr('x', 'y')).item():+.3f} on {sub.height} rows")

x and y show numeric association (maximum measured effect 0.546)
pooled Pearson r: 0.5150053176630283
within level A: r = +0.036 on 1006 rows
within level B: r = +0.000 on 994 rows


In [16]:
(alt.Chart(guard).mark_circle(size=16, opacity=0.3)
 .encode(x="x:Q", y="y:Q", color=alt.Color("g:N", title="Group"))
 .properties(title="A pooled correlation from group means alone", width=520, height=320))

alt.Chart(...)

What to look for: two round clouds, offset together along
the diagonal. Inside each cloud there is no slope — nothing to
reverse.

The pooled statement is strong and carries no qualification.
Its within-level correlations (+0.036 and +0.000) are the noise
the z bar is calibrated against: the combined within effect
must be measurably different from zero before the word
"reverses" is used, because a vanished association and a
reversed one call for opposite responses from the reader.

## Limits

The check reads the table it is given. A grouping that is not
recorded as a column cannot qualify a statement, and a grouping
recorded as a number is not a stratifier — stratifiers are
categorical columns, so a continuous confounder is out of reach
by construction.

Only the first qualifying stratifier, in column order, is
attached. A pair that reverses inside two dimensions is
reported with one of them.

Weak statements are not checked. Their tier already marks them
as below the size bar; the check is reserved for the statements
that read as settled.

The reversal the check certifies is the complete one: every
usable stratum against the pooled sign, with a measured within
effect. A pair whose direction flips in one large stratum and
holds in another is not qualified, and neither is a pair whose
within effects shrink toward zero without a measurable sign.
Both patterns are worth knowing about; neither is what the
qualification states, and the period-associations screen
([Associations that change by period](period_associations.ipynb))
covers the related case where the grouping is time.

The per-type field list for the associations table and the
`simpson` payload is in the findings reference
(https://github.com/fcela/onceover/blob/main/docs/findings.md).